# Project Results Summary

Very short notebook for quickly checking the final project results without rerunning model training.

## TL;DR

- The current production-style model is XGBoost trained on the `no_leakage` feature set.
- Test ROC-AUC is about `0.9325`, test F1 is about `0.7247`, and the selected business-cost threshold is `0.88`.
- The project also includes cross-validation, calibration, fairness, SHAP, leakage diagnostics, and CI smoke tests.

In [1]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
REPORTS = PROJECT_ROOT / "reports"

## Model Comparison

The table below uses the generated `reports/model_comparison.csv` file.

In [2]:
comparison = pd.read_csv(REPORTS / "model_comparison.csv")
cols = [
    "model",
    "feature_set",
    "test_roc_auc",
    "test_f1",
    "best_cost_threshold",
    "best_cost_business_cost",
]
comparison[cols].sort_values("test_roc_auc", ascending=False)

,model,feature_set,test_roc_auc,test_f1,best_cost_threshold,best_cost_business_cost
0,xgboost,no_leakage,0.9325,0.7247,0.88,1032.0
1,random_forest,no_leakage,0.9188,0.6706,0.84,1063.0
2,logistic_regression,no_leakage,0.8599,0.3339,0.92,1772.0


## Cross-Validation

A compact check that the headline model is not only strong on one split.

In [3]:
cv = pd.read_csv(REPORTS / "cross_validation_report.csv")
cv.sort_values("roc_auc_mean", ascending=False)

,model,roc_auc_mean,roc_auc_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std
0,xgboost,0.9348,0.0014,0.7657,0.0013,0.7505,0.0061,0.7816,0.0091
1,logistic_regression,0.8590,0.0013,0.6100,0.0006,0.5018,0.0035,0.7779,0.0066


## Calibration And Fairness

Calibration shows whether probabilities are usable; fairness gaps show where extra analysis is needed before production use.

In [4]:
calibration = json.loads((REPORTS / "calibration_curve.json").read_text())
fairness = pd.read_json(REPORTS / "fairness_gaps.json")

display(pd.DataFrame([
    {
        "brier_score": calibration["brier_score"],
        "log_loss": calibration["log_loss"],
    }
]))

fairness.sort_values("gap", ascending=False).head(5)

,brier_score,log_loss
0,0.0851,0.2844


,group_feature,metric,min,max,gap
10,person_home_ownership,recall,0.1429,0.8250,0.6821
11,person_home_ownership,f1,0.2500,0.8919,0.6419
1,age_band,recall,0.4880,0.7000,0.2120
9,person_home_ownership,approval_rate,0.0309,0.2222,0.1913
2,age_band,f1,0.6506,0.8000,0.1494


## Top SHAP Drivers

The most influential features according to the generated SHAP report.

In [5]:
pd.read_csv(REPORTS / "xgboost_shap_importance.csv").head(10)

,feature,mean_abs_shap
0,numeric__loan_int_rate,0.970916
1,numeric__loan_percent_income,0.820304
2,numeric__person_income,0.583096
3,categorical__person_home_ownership_RENT,0.361283
4,categorical__person_home_ownership_OWN,0.335311
5,categorical__loan_intent_VENTURE,0.318674
6,numeric__loan_amnt,0.163567
7,categorical__loan_intent_DEBTCONSOLIDATION,0.135045
8,categorical__loan_intent_HOMEIMPROVEMENT,0.100773
9,categorical__loan_intent_MEDICAL,0.097194


## Takeaway

The project is already a solid portfolio ML case: it has a reproducible pipeline, clear leakage handling, multiple validation reports, and a deployable demo. The main next improvement is deeper fairness analysis, especially for the largest subgroup gaps.